In [3]:
using System;
using System.Collections.Generic;
using System.Linq;

public enum OrderStatus
{
    Created,
    InProgress,
    Completed
}

public delegate void OrderStatusChangedHandler(Order order, OrderStatus newStatus);

public abstract class Person
{
    private string _firstName;
    private string _lastName;

    public string FirstName
    {
        get { return _firstName; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Имя не может быть пустым!");
            _firstName = value;
        }
    }

    public string LastName
    {
        get { return _lastName; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Фамилия не может быть пустой!");
            _lastName = value;
        }
    }

    public Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }

    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }

    public virtual void ShowInfo()
    {
        Console.WriteLine($"{GetFullName()}");
    }
}

public class Employee : Person
{
    private int _ordersProcessed;
    private string _position;

    public string Position
    {
        get { return _position; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Должность не может быть пустой!");
            _position = value;
        }
    }

    public int OrdersProcessed
    {
        get { return _ordersProcessed; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Количество обработанных заказов не может быть отрицательным!");
            _ordersProcessed = value;
        }
    }

    public Employee(string firstName, string lastName, string position) : base(firstName, lastName)
    {
        Position = position;
        OrdersProcessed = 0;
    }

    public void ProcessOrder()
    {
        OrdersProcessed++;
    }

    public void UpdateStatusNotify(Order order, OrderStatus newStatus)
    {
        Console.WriteLine($"Уведомление для сотрудника {GetFullName()}: статус заказа №{order.OrderId} изменён на {order.GetStatus()}");
    }

    public void ChangePosition(string newPosition)
    {
        Position = newPosition;
        Console.WriteLine($"Должность сотрудника {GetFullName()} изменена на: {Position}");
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Сотрудник: {GetFullName()}, Должность: {Position}, Обработано заказов: {OrdersProcessed}");
    }
}

public class Customer : Person
{
    private string _phoneNumber;

    public string PhoneNumber
    {
        get { return _phoneNumber; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Телефон не может быть пустым!");
            _phoneNumber = value;
        }
    }

    public Customer(string firstName, string lastName, string phoneNumber) : base(firstName, lastName)
    {
        PhoneNumber = phoneNumber;
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Заказчик: {GetFullName()}, Телефон: {PhoneNumber}");
    }

    public void UpdatePhone(string newPhone)
    {
        PhoneNumber = newPhone;
        Console.WriteLine($"Телефон заказчика {GetFullName()} обновлён: {PhoneNumber}");
    }

    public bool MatchesName(string lastName)
    {
        return string.Equals(LastName, lastName, StringComparison.OrdinalIgnoreCase);
    }
}

public class Order
{
    public int OrderId { get; private set; }
    private string _description;
    public Employee AssignedEmployee { get; private set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; private set; }
    public Customer Customer { get; private set; }

    public string Description
    {
        get { return _description; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Описание заказа не может быть пустым!");
            _description = value;
        }
    }

    public event OrderStatusChangedHandler StatusChanged;

    public Order(int orderId, string description, Customer customer)
    {
        OrderId = orderId;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void AssignEmployee(Employee employee)
    {
        if (Status == OrderStatus.Completed)
        {
            throw new InvalidOperationException("Нельзя назначить сотрудника на завершённый заказ");
        }

        if (AssignedEmployee != null)
        {
            StatusChanged -= AssignedEmployee.UpdateStatusNotify;
        }

        AssignedEmployee = employee;

        if (employee != null)
        {
            StatusChanged += employee.UpdateStatusNotify;
        }
    }

    public void ChangeStatus(OrderStatus newStatus)
    {
        OrderStatus oldStatus = Status;

        if (oldStatus == newStatus)
        {
            Console.WriteLine($"Заказ №{OrderId} уже в статусе {GetStatus()}");
            return;
        }

        bool validTransition = (oldStatus == OrderStatus.Created && newStatus == OrderStatus.InProgress) ||
            (oldStatus == OrderStatus.InProgress && newStatus == OrderStatus.Completed);

        if (!validTransition)
        {
            throw new InvalidOperationException($"Недопустимое изменение статуса заказа №{OrderId}: {oldStatus} на {newStatus}");
        }

        if (AssignedEmployee == null)
        {
            throw new InvalidOperationException($"Назначьте ответственного сотрудника на заказ №{OrderId}");
        }

        Status = newStatus;

        if (newStatus == OrderStatus.Completed)
        {
            AssignedEmployee.ProcessOrder();
        }

        StatusChanged?.Invoke(this, newStatus);
    }

    public static string GetStatusText(OrderStatus status)
    {
        if (status == OrderStatus.Created)
        {
            return "Создан";
        }
        else if (status == OrderStatus.InProgress)
        {
            return "В работе";
        }
        else if (status == OrderStatus.Completed)
        {
            return "Завершен";
        }
        else
        {
            return "Неизвестно";
        }
    }

    public string GetStatus()
    {
        return GetStatusText(Status);
    }

    public void ShowInfo()
    {
        string employeeName = "Не назначен";

        if (AssignedEmployee != null)
        {
            employeeName = AssignedEmployee.GetFullName();
        }

        Console.WriteLine($"Заказ №{OrderId}, Описание: {Description}, Статус: {GetStatus()}, Сотрудник: {employeeName}, Заказчик: {Customer.GetFullName()}, Создан: {CreatedAt:dd.MM.yyyy HH:mm}");
    }
}

public class Company
{
    private string _name;
    public List<Employee> Employees { get; private set; }
    public List<Order> Orders { get; private set; }
    public List<Customer> Customers { get; private set; }
    
    private int _nextOrderId;

    public string Name
    {
        get { return _name; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название компании не может быть пустым!");
            _name = value;
        }
    }

    public Company(string name)
    {
        Name = name;
        Employees = new List<Employee>();
        Orders = new List<Order>();
        Customers = new List<Customer>();
        _nextOrderId = 1;
    }

    public void AddEmployee(Employee employee)
    {
        if (!Employees.Contains(employee))
        {
            Employees.Add(employee);
            Console.WriteLine($"Сотрудник {employee.GetFullName()} добавлен в компанию {Name}");
        }
    }

    public void AddCustomer(Customer customer)
    {
        if (!Customers.Contains(customer))
        {
            Customers.Add(customer);
            Console.WriteLine($"Заказчик {customer.GetFullName()} добавлен в систему");
        }
    }

    public Order AddOrder(string description, Customer customer)
    {
        if (!Customers.Contains(customer))
        {
            throw new InvalidOperationException("Заказчик не зарегистрирован в компании");
        }

        Order order = new Order(_nextOrderId, description, customer);
        _nextOrderId++;
        Orders.Add(order);
        Console.WriteLine($"Заказ №{order.OrderId} создан для {customer.GetFullName()}");
        return order;
    }

    public void AssignEmployee(Order order, Employee employee)
    {
        if (!Orders.Contains(order))
        {
            throw new InvalidOperationException("Заказ не найден в компании");
        }

        if (!Employees.Contains(employee))
        {
            throw new InvalidOperationException("Сотрудник не найден в компании");
        }

        order.AssignEmployee(employee);
        Console.WriteLine($"На заказ №{order.OrderId} назначен сотрудник {employee.GetFullName()}");
    }

    public void ChangeOrderStatus(Order order, OrderStatus newStatus)
    {
        if (!Orders.Contains(order))
            throw new InvalidOperationException("Заказ не найден в компании");

        order.ChangeStatus(newStatus);
    }

    public List<Order> GetOrdersByStatus(OrderStatus status)
    {
        return Orders.Where(o => o.Status == status).ToList();
    }

    public List<Order> GetOrdersByCustomer(string customerName)
    {
        return Orders.Where(o => o.Customer.MatchesName(customerName)).ToList();
    }

    public void GetEmployeeReport()
    {
        Console.WriteLine($"-- Отчёт по сотрудникам компании '{Name}' --");

        if (Employees.Count == 0)
        {
            Console.WriteLine("Сотрудников нет");
            return;
        }

        foreach (var employee in Employees)
        {
            employee.ShowInfo();
        }
    }
}

Company company = new Company("АлгоритмПлюс");

Employee employee1 = new Employee("Виктория", "Вегерина", "Проджект-менеджер");
Employee employee2 = new Employee("Руслан", "Малыхин", "Fullstack-разработчик");
Employee employee3 = new Employee("Данил", "Казаков", "QA-инженер");
company.AddEmployee(employee1);
company.AddEmployee(employee2);
company.AddEmployee(employee3);
Console.WriteLine();
Customer customer1 = new Customer("Полина", "Ушакова", "89123456789");
Customer customer2 = new Customer("Богдан", "Горбунов", "89987654321");
Customer customer3 = new Customer("Илья", "Романов", "89991234567");
company.AddCustomer(customer1);
company.AddCustomer(customer2);
company.AddCustomer(customer3);
Console.WriteLine();
Order order1 = company.AddOrder("Разработка корпоративного сайта", customer1);
Order order2 = company.AddOrder("Создание мобильного приложения", customer2);
Order order3 = company.AddOrder("Тестирование корпоративного сайта", customer3);
Order order4 = company.AddOrder("Управление проектом разработки", customer1);
Console.WriteLine();

company.AssignEmployee(order1, employee2);
company.AssignEmployee(order2, employee2);
company.AssignEmployee(order3, employee3);
company.AssignEmployee(order4, employee1);

order1.StatusChanged += OnOrderStatusChanged;
order2.StatusChanged += OnOrderStatusChanged;
order3.StatusChanged += OnOrderStatusChanged;
order4.StatusChanged += OnOrderStatusChanged;

static void OnOrderStatusChanged(Order order, OrderStatus newStatus)
{
    Console.WriteLine($"Заказ №{order.OrderId} изменил статус на: {Order.GetStatusText(newStatus)}");
}

Console.WriteLine();
Console.WriteLine("--- Смена статусов ---");
company.ChangeOrderStatus(order1, OrderStatus.InProgress);
company.ChangeOrderStatus(order2, OrderStatus.InProgress);
company.ChangeOrderStatus(order1, OrderStatus.Completed);
company.ChangeOrderStatus(order3, OrderStatus.InProgress);
company.ChangeOrderStatus(order3, OrderStatus.Completed);
company.ChangeOrderStatus(order4, OrderStatus.InProgress);
company.ChangeOrderStatus(order4, OrderStatus.Completed);

Console.WriteLine();
Console.WriteLine("--- Все заказы ---");
foreach (var order in company.Orders)
{
    order.ShowInfo();
}

Console.WriteLine();
Console.WriteLine("--- Заказы в статусе: В работе ---");
var inProgress = company.GetOrdersByStatus(OrderStatus.InProgress);
if (inProgress.Count == 0)
{
    Console.WriteLine("Нет заказов в работе");
}
else
{
    foreach (var order in inProgress)
    {
        order.ShowInfo();
    }
}

Console.WriteLine();
Console.WriteLine("--- Заказы в статусе: Завершен ---");
var completed = company.GetOrdersByStatus(OrderStatus.Completed);
if (completed.Count == 0)
{
    Console.WriteLine("Нет завершённых заказов");
}
else
{
    foreach (var order in completed)
    {
        order.ShowInfo();
    }
}

Console.WriteLine();
Console.WriteLine("--- Поиск заказов по заказчику ---");
var customerOrders = company.GetOrdersByCustomer("Ушакова");
if (customerOrders.Count == 0)
{
    Console.WriteLine("Заказов для заказчика не найдено");
}
else
{
    Console.WriteLine($"Найдено заказов: {customerOrders.Count}");
    foreach (var order in customerOrders)
    {
        order.ShowInfo();
    }
}

company.GetEmployeeReport();

Сотрудник Виктория Вегерина добавлен в компанию АлгоритмПлюс
Сотрудник Руслан Малыхин добавлен в компанию АлгоритмПлюс
Сотрудник Данил Казаков добавлен в компанию АлгоритмПлюс

Заказчик Полина Ушакова добавлен в систему
Заказчик Богдан Горбунов добавлен в систему
Заказчик Илья Романов добавлен в систему

Заказ №1 создан для Полина Ушакова
Заказ №2 создан для Богдан Горбунов
Заказ №3 создан для Илья Романов
Заказ №4 создан для Полина Ушакова

На заказ №1 назначен сотрудник Руслан Малыхин
На заказ №2 назначен сотрудник Руслан Малыхин
На заказ №3 назначен сотрудник Данил Казаков
На заказ №4 назначен сотрудник Виктория Вегерина

--- Смена статусов ---
Уведомление для сотрудника Руслан Малыхин: статус заказа №1 изменён на В работе
Заказ №1 изменил статус на: В работе
Уведомление для сотрудника Руслан Малыхин: статус заказа №2 изменён на В работе
Заказ №2 изменил статус на: В работе
Уведомление для сотрудника Руслан Малыхин: статус заказа №1 изменён на Завершен
Заказ №1 изменил статус на: За